# Pipeline ML — GlicNutri (paciente-dia)

Este notebook consome o CSV **paciente-dia** por esta ordem: variável de ambiente `GLICNUTRI_CSV_PATH`; senão `machine-learning/data/glicnutri_patient_day_export.csv` (gerado por [`export_supabase_csv.py`](../scripts/export_supabase_csv.py)); senão o exemplo `machine-learning/data/sample_glicnutri_patient_day.csv`. Executa validação de schema, % de nulos, EDA resumida e quatro tarefas:

1. **Classificação** — dia com glicemia média elevada (rótulo: `glucose_mean_mg_dl >= 150`).
2. **Regressão** — prever `glucose_mean_mg_dl` a partir de hábitos/resumos diários.
3. **Clusterização** — grupos de dias por padrão nutricional/leituras.
4. **Similaridade** — vizinhos mais próximos no espaço de features (NearestNeighbors).

Ao final, os artefatos são gravados em `machine-learning/api/artifacts/` para uso pela API FastAPI (`POST /predict`). Para dados reais, gere o CSV com `machine-learning/scripts/export_supabase_csv.py` e aponte `CSV_PATH` abaixo.

In [ ]:
from pathlib import Path
import json
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while ROOT != ROOT.parent:
    if (ROOT / "machine-learning").is_dir() and (ROOT / "package.json").is_file():
        break
    ROOT = ROOT.parent
else:
    ROOT = Path.cwd()

_export_csv = ROOT / "machine-learning" / "data" / "glicnutri_patient_day_export.csv"
_sample_csv = ROOT / "machine-learning" / "data" / "sample_glicnutri_patient_day.csv"
_env_csv = os.environ.get("GLICNUTRI_CSV_PATH")
if _env_csv:
    CSV_PATH = Path(_env_csv)
elif _export_csv.exists():
    CSV_PATH = _export_csv
else:
    CSV_PATH = _sample_csv

ARTIFACTS = ROOT / "machine-learning" / "api" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    "n_leituras_glicemia",
    "carbs_sum_g",
    "kcal_sum",
    "protein_sum_g",
    "fat_sum_g",
    "n_refeicoes_ia",
    "n_eventos_medicacao",
]

print("ROOT:", ROOT)
print("CSV:", CSV_PATH.resolve())

In [ ]:
df = pd.read_csv(CSV_PATH)
df["dia"] = pd.to_datetime(df["dia"])

# --- Validação inicial (Semana 2: schema + qualidade mínima)
_required = ["patient_id", "dia", "glucose_mean_mg_dl"] + FEATURE_COLS
_missing_cols = [c for c in _required if c not in df.columns]
if _missing_cols:
    raise ValueError(f"Colunas em falta no CSV: {_missing_cols}")

_dup = df.duplicated(subset=["patient_id", "dia"]).sum()
print(f"Fonte: {CSV_PATH.resolve()} | Linhas: {len(df)} | Duplicados (patient_id+dia): {_dup}")
_null_pct = df[_required].isna().mean() * 100
print("% nulos por coluna (amostra):\n", _null_pct.round(2).to_string())

# Features numéricas
assert not df[FEATURE_COLS].isna().all().all(), "Dataset sem features numéricas"
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0)

# Linhas com alvo ausente não podem entrar em tarefas supervisionadas
_df_sup = df.dropna(subset=["glucose_mean_mg_dl"]).copy()
_df_sup["target_glucose_high"] = (_df_sup["glucose_mean_mg_dl"] >= 150).astype(int)
print(f"Linhas para treino supervisionado: {len(_df_sup)} (removidas {len(df) - len(_df_sup)} com glucose_mean_mg_dl nulo)")

_df_sup.describe()

## EDA (exploração rápida)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["glucose_mean_mg_dl"].hist(ax=axes[0], bins=12, color="#159365", edgecolor="white")
axes[0].set_title("Distribuição glicemia média (mg/dL)")
axes[1].scatter(df["carbs_sum_g"], df["glucose_mean_mg_dl"], alpha=0.7, c="#159365")
axes[1].set_xlabel("Carboidratos (g/dia)")
axes[1].set_ylabel("Glicemia média")
plt.tight_layout()
plt.show()

corr = df[["glucose_mean_mg_dl"] + FEATURE_COLS].corr()
corr["glucose_mean_mg_dl"].sort_values(ascending=False)

## Tarefa 1 — Classificação

In [ ]:
X = _df_sup[FEATURE_COLS]
y_cls = _df_sup["target_glucose_high"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_cls,
    test_size=0.25,
    random_state=42,
    stratify=y_cls if y_cls.nunique() > 1 else None,
)

clf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])
clf_pipe.fit(X_train, y_train)
pred = clf_pipe.predict(X_test)
print("Acurácia:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))

## Tarefa 2 — Regressão

In [ ]:
y_reg = _df_sup["glucose_mean_mg_dl"]
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.25, random_state=42)

reg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])
reg_pipe.fit(X_train, y_train)
pred_r = reg_pipe.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred_r))
print("R²:", r2_score(y_test, pred_r))

## Tarefa 3 — Clusterização

In [ ]:
k = min(3, len(_df_sup))
km = KMeans(n_clusters=k, random_state=42, n_init=10)
km.fit(X)
_df_sup["cluster"] = km.labels_
_df_sup["cluster"].value_counts()

## Tarefa 4 — Similaridade / recomendação por vizinhança

In [ ]:
nn = NearestNeighbors(n_neighbors=min(3, len(X)))
nn.fit(X)
dist, idx = nn.kneighbors(X.iloc[[0]], return_distance=True)
print("Distâncias:", dist)
print("Índices vizinhos:", idx)

## Persistência (joblib) — alinhado à API `POST /predict`

In [ ]:
joblib.dump(clf_pipe, ARTIFACTS / "model_classification.joblib")
joblib.dump(reg_pipe, ARTIFACTS / "model_regression.joblib")
joblib.dump(km, ARTIFACTS / "model_clustering.joblib")
joblib.dump(nn, ARTIFACTS / "model_similarity.joblib")
joblib.dump(X.values, ARTIFACTS / "similarity_train_matrix.joblib")

meta = {
    "feature_columns": FEATURE_COLS,
    "target_classification": "target_glucose_high (glucose_mean_mg_dl >= 150)",
    "target_regression": "glucose_mean_mg_dl",
    "csv_source": str(CSV_PATH.resolve()),
    "supervised_rows": int(len(_df_sup)),
    "dropped_rows_missing_glucose_mean": int(len(df) - len(_df_sup)),
}
with open(ARTIFACTS / "training_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("Artefatos salvos em", ARTIFACTS)